# Procesamiento de CSV - Unión de Datasets de Inmuebles

Este notebook une el dataset principal de inmuebles con datos macro de cada país como PIB, desigualdad, población y crimen.

**Objetivo:** Enriquecer la información de cada propiedad con contexto demográfico, económico y de seguridad del país.

In [2]:
import pandas as pd
import os

**Importaciones:**
- `pandas`: librería para leer y manipular datos en formato CSV y tablas
- `os`: librería para trabajar con rutas de archivos en el sistema

In [3]:
COUNTRY_MAP = {
    'united states of america': 'united states',
    'usa': 'united states',
    'us': 'united states',
    'uae': 'united arab emirates',
    'north macedonia': 'north macedonia',
    'macedonia fyrom': 'north macedonia',
    'korea south': 'south korea',
    'south korea': 'south korea',
    'korea north': 'north korea',
    'ivory coast': 'cote divoire',
    'cote divoire': 'cote divoire',
    'viet nam': 'vietnam',
    'lao pdr': 'laos',
    'bolivia plurinational state of': 'bolivia',
    'venezuela bolivarian republic of': 'venezuela',
    'united kingdom': 'united kingdom',
    'uk': 'united kingdom',
    'russian federation': 'russia',
    'czechia': 'czech republic',
    'brunei darussalam': 'brunei',
    'iran islamic republic of': 'iran',
    'palestinian territories': 'palestine',
    'north cyprus': 'northern cyprus',
}

**Diccionario de normalización de países:**

Este diccionario traduce nombres de países con variaciones a un nombre estándar. Por ejemplo:
- 'USA' se normaliza a 'united states'
- 'UAE' se normaliza a 'united arab emirates'

Esto es necesario porque cada archivo CSV usa nombres diferentes para el mismo país.

In [4]:
def clean_country(name):
    if not isinstance(name, str):
        return ''
    
    value = name.strip().lower()
    value = value.replace("'", "")
    value = value.replace('`', '')
    value = value.replace('(', ' ')
    value = value.replace(')', ' ')
    value = value.replace('[', ' ')
    value = value.replace(']', ' ')
    value = value.replace(',', ' ')
    value = ' '.join(value.split())
    
    return COUNTRY_MAP.get(value, value)

**Función `clean_country()`:**

Esta función limpia un nombre de país:
1. Convierte a minúsculas
2. Elimina caracteres especiales como paréntesis, corchetes, comas
3. Busca en el diccionario si existe un nombre normalizado
4. Si no encuentra, devuelve el nombre limpio tal cual

Ejemplo: "United States (USA)" → "united states"

In [6]:
def read_country_file(path, country_col, keep_cols):
    with open(path, encoding='utf-8', errors='replace') as fh:
        df = pd.read_csv(fh)
    
    df['country_norm'] = df[country_col].map(clean_country)
    keep = [country_col] + keep_cols + ['country_norm']
    
    return df[keep]

**Función `read_country_file()`:**

Lee un archivo CSV y normaliza los nombres de país:
- `path`: ruta del archivo CSV
- `country_col`: nombre de la columna que contiene el país
- `keep_cols`: list de columnas que queremos guardar

Pasos:
1. Abre y lee el CSV
2. Crea una nueva columna `country_norm` aplicando `clean_country()` a cada fila
3. Devuelve solo las columnas que nos interesan

In [7]:
with open('../../Data/Raw/world_real_estate_data(147k).csv', encoding='utf-8', errors='replace') as fh:
    real_estate = pd.read_csv(fh)

real_estate['country_norm'] = real_estate['country'].map(clean_country)

print('Dataset de inmuebles cargado:')
print('Filas:', len(real_estate))
print('Países únicos:', real_estate['country_norm'].nunique())
print('Columnas:', real_estate.columns.tolist())

Dataset de inmuebles cargado:
Filas: 147536
Países únicos: 28
Columnas: ['title', 'country', 'location', 'building_construction_year', 'building_total_floors', 'apartment_floor', 'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms', 'apartment_total_area', 'apartment_living_area', 'price_in_USD', 'image', 'url', 'country_norm']


**Carga del dataset principal:**

Leemos el archivo de inmuebles desde la carpeta `Raw`:
- 147,536 propiedades
- 28 países
- Información: ubicación, precio, características, construcción, etc.

Se crea la columna `country_norm` para poder unir después con otros datasets.

In [8]:
gdp = read_country_file('../../Data/Raw/countries_by_gpd_(ppp).csv', 'Country/Territory', ['(millions of current Int$)'])
gdp = gdp.rename(columns={'(millions of current Int$)': 'gdp_ppp'})

print('Dataset GDP cargado:', len(gdp), 'países')
print(gdp.head())

Dataset GDP cargado: 197 países
     Country/Territory     gdp_ppp        country_norm
0                China  23,460,170               china
1        United States  21,427,700       united states
2  European Union[n 2]  20,794,743  european union n 2
3                India   9,611,679               india
4                Japan   5,459,155               japan


**Carga de datos de PIB (GDP):**

- `gdp_ppp`: PIB ajustado por paridad de poder adquisitivo
- Se renombra la columna para que sea más clara
- 197 países

In [9]:
gini = read_country_file('../../Data/Raw/countries_by_income_equality.csv', 'Country', ['World Bank\ufffdGini', 'Year'])
gini = gini.rename(columns={'World Bank\ufffdGini': 'gini', 'Year': 'gini_year'})

print('Dataset Gini cargado:', len(gini), 'países')
print(gini.head())

Dataset Gini cargado: 177 países
       Country  gini  gini_year country_norm
0  Afghanistan   NaN        NaN  afghanistan
1      Albania  33.2     2017.0      albania
2      Algeria  27.6     2011.0      algeria
3       Angola  51.3     2018.0       angola
4    Argentina  41.4     2018.0    argentina


**Carga de datos de desigualdad (Gini):**

- `gini`: Coeficiente de Gini (0-100, más alto = más desigualdad)
- `gini_year`: Año de los datos
- 177 países

In [10]:
pop = read_country_file('../../Data/Raw/countries_by_population_(united nations).csv', 'Country/Territory', ['UN continental', 'UN statistical', 'Population (2018)', 'Population (2019)', 'Change'])
pop = pop.rename(columns={
    'UN continental': 'un_continent',
    'UN statistical': 'un_statistical',
    'Population (2018)': 'population_2018',
    'Population (2019)': 'population_2019',
    'Change': 'population_change',
})

print('Dataset Población cargado:', len(pop), 'países')
print(pop.head())

Dataset Población cargado: 233 países
  Country/Territory un_continent      un_statistical population_2018  \
0             China         Asia        Eastern Asia   1,427,647,786   
1             India         Asia       Southern Asia   1,352,642,280   
2     United States     Americas    Northern America     327,096,265   
3         Indonesia         Asia  South-eastern Asia     267,670,543   
4          Pakistan         Asia       Southern Asia     212,228,286   

  population_2019 population_change   country_norm  
0   1,433,783,686             0.43%          china  
1   1,366,417,754             1.02%          india  
2     329,064,917             0.60%  united states  
3     270,625,568             1.10%      indonesia  
4     216,565,318             2.04%       pakistan  


**Carga de datos de población:**

- `un_continent`: Continente según ONU
- `population_2018` y `population_2019`: Población en esos años
- `population_change`: Porcentaje de cambio año a año
- 233 países/territorios

In [11]:
crime = read_country_file('../../Data/Raw/crime_index_by_country.csv', 'Country', ['Crime Index', 'Safety Index'])
crime = crime.rename(columns={'Crime Index': 'crime_index', 'Safety Index': 'safety_index'})

print('Dataset Crimen cargado:', len(crime), 'países')
print(crime.head())

Dataset Crimen cargado: 133 países
            Country  crime_index  safety_index      country_norm
0         Venezuela        84.36         15.64         venezuela
1  Papua New Guinea        80.04         19.96  papua new guinea
2      South Africa        77.29         22.71      south africa
3       Afghanistan        76.97         23.03       afghanistan
4          Honduras        76.65         23.35          honduras


**Carga de datos de seguridad:**

- `crime_index`: Índice de crimen (0-100, más alto = más crimen)
- `safety_index`: Índice de seguridad (100 - crime_index)
- 133 países

In [12]:
unemp = read_country_file('../../Data/Raw/unemployment_rate_by_country.csv', 'Country', ['Last', 'Previous', 'Reference', 'Unit'])
unemp = unemp.rename(columns={
    'Last': 'unemployment_last',
    'Previous': 'unemployment_previous',
    'Reference': 'unemployment_reference',
    'Unit': 'unemployment_unit',
})

print('Dataset Desempleo cargado:', len(unemp), 'países')
print(unemp.head())

Dataset Desempleo cargado: 181 países
                  Country  unemployment_last  unemployment_previous  \
0  Bosnia and Herzegovina              34.85                  34.67   
1                  Angola              34.00                  32.70   
2                 Namibia              33.40                  34.00   
3               Palestine              28.50                  26.50   
4                 Nigeria              27.10                  23.10   

  unemployment_reference unemployment_unit            country_norm  
0                 20-Aug                 %  bosnia and herzegovina  
1                 20-Sep                 %                  angola  
2                 18-Dec                 %                 namibia  
3                 20-Sep                 %               palestine  
4                 20-Jun                 %                 nigeria  


**Carga de datos de desempleo:**

- `unemployment_last`: Tasa de desempleo actual
- `unemployment_previous`: Tasa anterior
- `unemployment_reference`: Fecha de referencia
- 181 países

In [13]:
lat = read_country_file('../../Data/Raw/countries_by_latitude.csv', 'name', ['country', 'latitude', 'longitude'])
lat = lat.rename(columns={'country': 'country_code', 'name': 'country_name'})

print('Dataset Ubicación cargado:', len(lat), 'países')
print(lat.head())

Dataset Ubicación cargado: 245 países
           country_name country_code   latitude  longitude  \
0               Andorra           AD  42.546245   1.601554   
1  United Arab Emirates           AE  23.424076  53.847818   
2           Afghanistan           AF  33.939110  67.709953   
3   Antigua and Barbuda           AG  17.060816 -61.796428   
4              Anguilla           AI  18.220554 -63.068615   

           country_norm  
0               andorra  
1  united arab emirates  
2           afghanistan  
3   antigua and barbuda  
4              anguilla  


**Carga de datos geográficos:**

- `country_code`: Código ISO de 2 letras (p.ej., 'ES' para España)
- `latitude` y `longitude`: Coordenadas
- 245 países/territorios

Estos datos permitirán hacer mapas interactivos más tarde.

In [14]:
codes = read_country_file('../../Data/Raw/plotly_countries_and_codes.csv', 'COUNTRY', ['CODE'])
codes = codes.rename(columns={'CODE': 'country_iso_code'})

print('Dataset Códigos cargado:', len(codes), 'países')
print(codes.head())

Dataset Códigos cargado: 222 países
          COUNTRY country_iso_code    country_norm
0     Afghanistan              AFG     afghanistan
1         Albania              ALB         albania
2         Algeria              DZA         algeria
3  American Samoa              ASM  american samoa
4         Andorra              AND         andorra


**Carga de códigos ISO de 3 letras:**

- `country_iso_code`: Código ISO de 3 letras (p.ej., 'ESP' para España)
- Útil para visualizaciones en Plotly
- 222 países

In [15]:
merged = real_estate.merge(gdp, on='country_norm', how='left')
print('Después de unir GDP:', len(merged), 'filas')

merged = merged.merge(gini, on='country_norm', how='left')
print('Después de unir Gini:', len(merged), 'filas')

merged = merged.merge(pop, on='country_norm', how='left')
print('Después de unir Población:', len(merged), 'filas')

merged = merged.merge(crime, on='country_norm', how='left')
print('Después de unir Crimen:', len(merged), 'filas')

merged = merged.merge(unemp, on='country_norm', how='left')
print('Después de unir Desempleo:', len(merged), 'filas')

Después de unir GDP: 147536 filas
Después de unir Gini: 147536 filas
Después de unir Población: 147536 filas
Después de unir Crimen: 147536 filas
Después de unir Desempleo: 147536 filas


**Unión secuencial de datasets:**

Usamos `merge(..., how='left')` para:
- Mantener todas las filas del dataset principal (inmuebles)
- Añadir información de cada país si existe
- Si un país no tiene cierta información, queda vacío (NaN)

Se verifica que no se pierdan filas en el proceso.

In [16]:
merged = merged.merge(lat[['country_norm', 'country_code', 'latitude', 'longitude']], on='country_norm', how='left')
print('Después de unir Ubicación:', len(merged), 'filas')

merged = merged.merge(codes[['country_norm', 'country_iso_code']], on='country_norm', how='left')
print('Después de unir Códigos ISO:', len(merged), 'filas')

Después de unir Ubicación: 147536 filas
Después de unir Códigos ISO: 147536 filas


**Unión final de datos geográficos:**

Añadimos:
- Coordenadas para mapas
- Códigos ISO para visualizaciones de Plotly

In [17]:
merged.to_csv('../../Data/merged_complete.csv', index=False, encoding='utf-8')
print('Archivo guardado: merged_complete.csv')

Archivo guardado: merged_complete.csv


**Guardado del archivo:**

- Se guarda el dataset completo en formato CSV
- Se usa `index=False` para no guardar índices
- Archivo resultante: `merged_complete.csv`

In [18]:
unmatched = merged[merged['gdp_ppp'].isna() & merged['country_norm'].notna()]['country']
unmatched = [str(x).strip() for x in unmatched.unique() if pd.notna(x)]

print('Resumen final:')
print('- Total de filas:', len(merged))
print('- Países únicos:', merged['country_norm'].nunique())
print('- Países sin datos de GDP:', unmatched)

Resumen final:
- Total de filas: 147536
- Países únicos: 28
- Países sin datos de GDP: ['Northern Cyprus']


**Validación final:**

Verifica qué países en el dataset de inmuebles no tuvieron coincidencia con datos macro:
- 147,536 propiedades
- 28 países representados
- Algunos países pueden no tener todos los datos (GDP, Gini, etc.)

In [19]:
print('\nMuestra del dataset final:')
print(merged.head())
print('\nColumnas disponibles:')
print(merged.columns.tolist())


Muestra del dataset final:
                                               title   country  \
0  2 room apartment 120 m² in Mediterranean Regio...    Turkey   
1              4 room villa 500 m² in Kalkan, Turkey    Turkey   
2          1 room apartment 65 m² in Antalya, Turkey    Turkey   
3             1 room apartment  in Pattaya, Thailand  Thailand   
4             2 room apartment  in Pattaya, Thailand  Thailand   

                                    location  building_construction_year  \
0               Mediterranean Region, Turkey                         NaN   
1  Kalkan, Mediterranean Region, Kas, Turkey                      2021.0   
2      Mediterranean Region, Antalya, Turkey                         NaN   
3      Chon Buri Province, Pattaya, Thailand                      2020.0   
4      Chon Buri Province, Pattaya, Thailand                      2026.0   

   building_total_floors  apartment_floor  apartment_rooms  \
0                    5.0              1.0              3

**Vista previa del resultado:**

Muestra las primeras filas y todas las columnas disponibles para verificar que la unión fue correcta.

**Próximos pasos:**
- Limpiar valores nulos
- Explorar correlaciones
- Visualizar en Streamlit o Power BI

## Limpieza de valores nulos

En este paso limpiamos nulos de forma simple:

1. Convertimos a numéricas las columnas que deberían ser numéricas.
2. Quitamos filas sin precio o sin país normalizado.
3. Rellenamos nulos numéricos con mediana por país y, si falta, mediana global.
4. Rellenamos nulos de texto con `unknown`.

El resultado se guarda en `processed.csv` dentro de esta misma carpeta (`pre-procesamiento`).

In [20]:
# Copia de trabajo
cleaned = merged.copy()

# 1) Convertir columnas numéricas
numeric_cols = [
    'building_construction_year', 'building_total_floors', 'apartment_floor',
    'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms',
    'price_in_USD', 'latitude', 'longitude', 'gini', 'gini_year',
    'crime_index', 'safety_index', 'unemployment_last', 'unemployment_previous'
]

for col in numeric_cols:
    if col in cleaned.columns:
        cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce')

# Limpiar áreas (vienen como texto, por ejemplo: "120 m²")
for col in ['apartment_total_area', 'apartment_living_area']:
    if col in cleaned.columns:
        cleaned[col] = (
            cleaned[col]
            .astype(str)
            .str.replace('m²', '', regex=False)
            .str.replace('m2', '', regex=False)
            .str.extract(r'([0-9]+(?:\.[0-9]+)?)')[0]
        )
        cleaned[col] = pd.to_numeric(cleaned[col], errors='coerce')

# Limpiar numéricos macro en formato texto
if 'gdp_ppp' in cleaned.columns:
    cleaned['gdp_ppp'] = pd.to_numeric(cleaned['gdp_ppp'].astype(str).str.replace(',', '', regex=False), errors='coerce')

for col in ['population_2018', 'population_2019']:
    if col in cleaned.columns:
        cleaned[col] = pd.to_numeric(cleaned[col].astype(str).str.replace(',', '', regex=False), errors='coerce')

if 'population_change' in cleaned.columns:
    cleaned['population_change'] = pd.to_numeric(cleaned['population_change'].astype(str).str.replace('%', '', regex=False), errors='coerce')

# 2) Eliminar filas críticas sin precio o sin país
before_rows = len(cleaned)
cleaned = cleaned.dropna(subset=['country_norm', 'price_in_USD'])

# 3) Rellenar nulos numéricos
fill_numeric_cols = [
    'building_construction_year', 'building_total_floors', 'apartment_floor',
    'apartment_rooms', 'apartment_bedrooms', 'apartment_bathrooms',
    'apartment_total_area', 'apartment_living_area', 'gdp_ppp', 'gini',
    'crime_index', 'safety_index', 'unemployment_last', 'unemployment_previous',
    'population_2018', 'population_2019', 'population_change', 'latitude', 'longitude'
]

for col in fill_numeric_cols:
    if col in cleaned.columns:
        cleaned[col] = cleaned.groupby('country_norm')[col].transform(lambda s: s.fillna(s.median()))
        cleaned[col] = cleaned[col].fillna(cleaned[col].median())

# 4) Rellenar nulos de texto
fill_text_cols = ['country_code', 'country_iso_code', 'un_continent', 'un_statistical', 'unemployment_reference', 'unemployment_unit']
for col in fill_text_cols:
    if col in cleaned.columns:
        cleaned[col] = cleaned[col].fillna('unknown')

# Guardar resultado en la carpeta actual del notebook (no en Data)
output_path = 'processed.csv'
cleaned.to_csv(output_path, index=False, encoding='utf-8')

print('Limpieza de nulos completada')
print('Filas antes:', before_rows)
print('Filas después:', len(cleaned))
print('Archivo guardado:', output_path)

nulls_after = cleaned.isna().sum().sort_values(ascending=False)
print('\nTop 10 columnas con más nulos tras limpieza:')
print(nulls_after.head(10))

Limpieza de nulos completada
Filas antes: 147536
Filas después: 144961
Archivo guardado: processed.csv

Top 10 columnas con más nulos tras limpieza:
Country                       875
Country_y                     875
Country/Territory_x           875
Country_x                     875
Country/Territory_y           875
gini_year                     875
location                      131
country                       130
building_construction_year      0
title                           0
dtype: int64


## Vista rápida de `processed.csv`

Mostramos los primeros 20 registros del archivo procesado para validar nombres de columnas y contenido.

In [ ]:
processed_preview = pd.read_csv('processed.csv')
print('Shape:', processed_preview.shape)

# Mostrar solo 8 columnas clave
cols_visibles = [
    'title',
    'country',
    'location',
    'price_in_USD',
    'apartment_total_area',
    'apartment_bedrooms',
    'crime_index',
    'unemployment_last'
]

display(processed_preview[cols_visibles].head(20))

Shape: (144961, 38)


,title,country,location,building_construction_year,building_total_floors,apartment_floor,apartment_rooms,apartment_bedrooms,apartment_bathrooms,apartment_total_area,apartment_living_area,price_in_USD,image,url,country_norm,Country/Territory_x,gdp_ppp,Country_x,gini,gini_year,Country/Territory_y,un_continent,un_statistical,population_2018,population_2019,population_change,Country_y,crime_index,safety_index,Country,unemployment_last,unemployment_previous,unemployment_reference,unemployment_unit,country_code,latitude,longitude,country_iso_code
0,"2 room apartment 120 m² in Mediterranean Region, Turkey",Turkey,"Mediterranean Region, Turkey",2023.0,5.0,1.0,3.0,2.0,2.0,120.0,110.0,315209.0,https://realting.com/uploads/bigSlider/ab3/888989d8c45280f0e1e17d633ee94.webp,https://realting.com/property-for-sale/turkey/inest-homes/1766770,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
1,"4 room villa 500 m² in Kalkan, Turkey",Turkey,"Kalkan, Mediterranean Region, Kas, Turkey",2021.0,2.0,3.0,3.0,2.0,1.0,500.0,480.0,1108667.0,https://realting.com/uploads/bigSlider/87b/67960b9229216decf200460d37dca.webp,https://realting.com/property-for-sale/turkey/inest-homes/1767446,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
2,"1 room apartment 65 m² in Antalya, Turkey",Turkey,"Mediterranean Region, Antalya, Turkey",2023.0,5.0,2.0,2.0,1.0,1.0,65.0,60.0,173211.0,https://realting.com/uploads/bigSlider/030/a115091710bbd90676afae0ffb1df.webp,https://realting.com/property-for-sale/turkey/fsbo/1766776,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
3,"1 room apartment in Pattaya, Thailand",Thailand,"Chon Buri Province, Pattaya, Thailand",2020.0,15.0,5.0,2.0,1.0,1.0,47.0,40.0,99900.0,https://realting.com/uploads/bigSlider/e9a/e061e09c8544dd3e449393b82e3b8.webp,https://realting.com/property-for-sale/thailand/alza-real-estate/1754337,thailand,Thailand,1338781.0,Thailand,36.4,2018.0,Thailand,Asia,South-eastern Asia,68863514.0,69037513.0,0.25,Thailand,40.01,59.99,Thailand,1.90,2.10,20-Aug,%,TH,15.870032,100.992541,THA
4,"2 room apartment in Pattaya, Thailand",Thailand,"Chon Buri Province, Pattaya, Thailand",2026.0,8.0,3.0,3.0,2.0,1.0,47.0,36.0,67000.0,https://realting.com/uploads/bigSlider/453/aa2f3e8c7387dd5f91d4271ecc4e0.webp,https://realting.com/property-for-sale/thailand/alza-real-estate/1754327,thailand,Thailand,1338781.0,Thailand,36.4,2018.0,Thailand,Asia,South-eastern Asia,68863514.0,69037513.0,0.25,Thailand,40.01,59.99,Thailand,1.90,2.10,20-Aug,%,TH,15.870032,100.992541,THA
5,"1 room apartment 28 m² in Batumi, Georgia",Georgia,"Abkhazia, Batumi, Georgia",2026.0,15.0,7.0,1.0,1.0,1.0,28.0,48.0,35622.0,https://realting.com/uploads/bigSlider/16f/406c7a33e86a7ca2f5285bf817e47.webp,https://realting.com/property-for-sale/georgia/flatiko-llc/1765932,georgia,Georgia,58174.0,Georgia,36.4,2018.0,Georgia,Asia,Western Asia,4002942.0,3996765.0,0.09,Georgia,20.50,79.50,Georgia,12.70,13.90,18-Dec,%,GE,42.315407,43.356892,GEO
6,"4 room apartment 245 m² in Yesiloez, Turkey",Turkey,"Yesiloez, Mediterranean Region, Alanya, Turkey",2007.0,2.0,3.0,5.0,4.0,4.0,245.0,245.0,274415.0,https://realting.com/uploads/bigSlider/9c8/6f1d74ccdc5e0d8153dbcbb49d147.webp,https://realting.com/property-for-sale/turkey/fsbo/1732620,turkey,Turkey,2325617.0,Turkey,41.9,2018.0,Turkey,Asia,Western Asia,82340088.0,83429615.0,1.32,Turkey,39.50,60.50,Turkey,13.20,13.40,20-Aug,%,TR,38.963745,35.243322,TUR
7,"1 room studio apartment in Pattaya, Thailand",Thailand,"Chon Buri Province, Pattaya, Thailand",2026.0,8.0,2.0,1.0,1.0,1.0,47.0,25.0,44700.0,https://realting.com/uploads/bigSlider/b50/f4bbb75591310bcd8fa4a68e2564e.webp,https://realting.com/pro